## Import Python libraries

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## User defined inputs

In [ ]:
##### --------------------------
# User input: data - paths and labels
##### --------------------------

# alias for paths to csv files
path_to_train      = r"data_csv/orders_train.csv"
path_to_validation = r"data_csv/orders_validation.csv"
path_to_test       = r"data_csv/orders_test.csv"

# alias for columns of the csv files
id_name = "order_id"           # unique id
x_name = "item_count"          # first variable 
y_name = "preparation_minutes" # second variable

# names to be displayed in the plots
dic_label_name = {
    id_name: "Article ID",
    x_name: "Number of articles",    # label for x-axis
    y_name: "Preparation (minutes)", # label for y-axis
}

dic_data_unities = {
    id_name: "(id)",
    x_name: "(counts)",
    y_name: "(minutes)"
}

# Title for scatter plot of training data
train_title = "Scatter plot of the training CSV"


In [ ]:
# Source - https://stackoverflow.com/a/20079807
# Posted by codegeek, modified by community. See post 'Timeline' for change history
# Retrieved 2026-10-11, License - CC BY-SA 3.0

empty_dict = dict.fromkeys(['apple','ball'])
# empty_dict = {'apple': None, 'ball': None}

default_value = 'xyz'
nonempty_dict = dict.fromkeys(['apple','ball'],default_value)

# Or if None is a legitimate value for one or more of your key-value tuples, 
# you can use something like "SENTINEL = object()" and use SENTINEL as 
# your sentinel value, testing with "is" instead of "==".

In [ ]:
##### --------------------------
# Check input paths
##### --------------------------

print(path_to_train)
print(path_to_validation)
print(path_to_test)

assert Path(path_to_train).exists()
assert Path(path_to_validation).exists
assert Path(path_to_test).exists

data_csv/orders_train.csv
data_csv/orders_validation.csv
data_csv/orders_test.csv


In [234]:
##### --------------------------
# User input: lambda functions of the fitting model
##### --------------------------

fit_model = \
    lambda x, y, degree: \
        np.polynomial.Polynomial.fit(x, y, degree)
do_fit = \
    lambda x, y: {
        degree: fit_model(x, y, degree)
        for degree in degrees
    }

# usage : once data loaded
# models = do_fit(x_train, y_train) 

In [235]:
##### --------------------------
# User input: Parameters for the fit
##### --------------------------

# degrees used in Polynomial.fit
degrees = [0, 1, 2, 4]#, 12] 

# labels for the fitting plot
fit_labels = {
    0: "Constante", 
    1: "Droite", 
    2: "Degré 2", 
    4: "Degré 4",
    12: "Degré 12",
    }

# colors for the fitting plots
fit_colors = {
    0:  "#FF5733", 
    1:  "#23d4b7", 
    2:  "#af76f0",
    4:  "#058e30",
    12: "#ff9969",
}

# title for the fitting plot on training data
fit_title = "Fit of polynomial models on training data"

# bounds for plot of fit models
# usage: 
# x_grid = np.linspace(fit_xmin, fit_xmax, fit_grid_nb)
# for degree, model in models.items():
#     fig.add_scatter(x=x_grid, y=model(x_grid)

fit_xmin = None # if None, x_train.min() taken
fit_xmax = None # if None, y_train.max() taken
fit_grid = 400  # number of points for evaluation of models

In [236]:
# parameters for validation plots

val_title = "Validation plot"
val_nrows = None
val_ncols = None

val_xmin = None
val_xmax = None

val_ymin = None
val_ymax = None

val_xticks = None
val_xticks_target = 5
val_yticks = None
val_yticks_target = 5


In [237]:
# parameters for test plots

test_title = "Final test"
test_nrows = None
test_ncols = None

test_xmin = None
test_xmax = None

test_ymin = None
test_ymax = None

test_xticks = None
test_xticks_target = 5
test_yticks = None
test_yticks_target = 5

## Code

In [238]:
##### --------------------------
# Functions
##### --------------------------

def mae(observed, predicted):
    # mean absolute error
    return float(np.mean(np.abs(np.asarray(observed) - np.asarray(predicted))))

def lsm(observed, predicted):
    # least squared method
    return float(np.sqrt(np.asarray(observed)**2 - np.asarray(predicted)**2))

In [239]:
# get spacing for tick values of x, y in validation plot

def get_spacing(xmin, xmax, target_ticks=6):
    span = xmax - xmin
    if span <= 0:
        raise ValueError("xmax must be greater than xmin")

    raw_spacing = span / target_ticks
    magnitude = 10 ** np.floor(np.log10(raw_spacing))
    candidates = np.array([1, 2, 5, 10]) * magnitude

    return candidates[np.argmin(np.abs(candidates - raw_spacing))]


def get_ticks(xmin, xmax, target_ticks=6):
    spacing = get_spacing(xmin, xmax, target_ticks)

    start = np.ceil(xmin / spacing) * spacing
    stop = np.floor(xmax / spacing) * spacing

    return np.arange(start, stop + spacing * 0.5, spacing)

# usage :
# if val_xticks is None: val_xticks = get_ticks(val_xmin, val_xmax)


In [240]:
def add_prediction_view(
        figure, frame, x, observed, model, grid, color, row=1, col=1
    ):
    """Draw every observation and its vertical prediction error, without fitting."""
    predicted = model(x)
    segment_x, segment_y = [], []
    for item_count, actual, estimate in zip(x, observed, predicted):
            segment_x.extend([item_count, item_count, None])
            segment_y.extend([actual, estimate, None])
    figure.add_trace(go.Scatter(
        x=segment_x, y=segment_y, mode="lines",
        line={"color": "rgba(105,120,140,0.35)", "width": 1},
        hoverinfo="skip", showlegend=False), row=row, col=col)
    
    figure.add_trace(go.Scatter(
        x=grid, y=model(grid), mode="lines",
        line={"color": color, "width": 3},
        name="Prédiction", showlegend=False), row=row, col=col)
    
    details = np.column_stack([predicted, np.abs(observed - predicted)])
    figure.add_trace(go.Scatter(
        x=x, 
        y=observed, 
        mode="markers", 
        customdata=details,
        text=frame["order_id"], 
        marker={"size": 5, "color": "#22364c", "opacity": 0.65},
        hovertemplate=(
            "%{text}" + 
            "<br>Articles : %{x}"
            "<br>Observé : %{y:.3f}"
            "<br>Prédit : %{customdata[0]:.3f}"
            "<br>Erreur absolue : %{customdata[1]:.3f}<extra></extra>"
        ),
        showlegend=False), row=row, col=col)


In [241]:
l=[]
l.extend([1,1,None])
l.extend([2,2,None])

print(l)

[1, 1, None, 2, 2, None]


### Import datas

In [242]:
##### --------------------------
# Import and small checks CSV files
##### --------------------------

# load csv files
train = pd.read_csv(path_to_train)
validation = pd.read_csv(path_to_validation)
test = pd.read_csv(path_to_test)

# run some checks: uniqueness of order_id values
assert train[id_name].is_unique 
assert validation[id_name].is_unique
assert test[id_name].is_unique

# run some checks: disjoint order_id values between files
assert set(train[id_name]).isdisjoint(validation[id_name])
assert set(test[id_name]).isdisjoint(validation[id_name])
assert set(test[id_name]).isdisjoint(train[id_name])

In [243]:
##### --------------------------
# Scatter plot of training data
##### --------------------------

# Scatter plot of the training CSV
fig = px.scatter(
    train, 
    x=x_name, 
    y=y_name, 
    hover_data=[id_name],
    opacity=0.8,
    labels=dic_label_name,
    title=train_title)
fig.show()

### Polynomial fits

In [244]:
##### --------------------------
# Fit of the model
##### --------------------------

# pd.Series to NumPy for fit
x_train = train[x_name].to_numpy()
y_train = train[y_name].to_numpy()

# Fit: applying user_defined do_fit 
models = do_fit(x_train, y_train) 


##### --------------------------
# Plot
##### --------------------------

# array to evaluate the models
if fit_xmin is None: fit_xmin = np.floor(x_train.min())
if fit_xmax is None: fit_xmax = np.ceil(x_train.max())
x_grid = np.linspace(fit_xmin, fit_xmax, fit_grid)

# plot of training data
fig = px.scatter(
    train, 
    x=x_name, 
    y=y_name, 
    hover_data=[id_name],
    opacity=0.8,
    labels=dic_label_name,
    title=train_title + f" — {len(x_train)} data points")

# add of the models
for degree, model in models.items():
    fig.add_scatter(
        x=x_grid, 
        y=model(x_grid), 
        mode="lines", 
        line_color=fit_colors[degree],
        name=fit_labels[degree])

fig.show()

In [ ]:
# Validation plots

nb_plots = len(models)

if val_ncols is None: val_ncols = int(np.ceil(np.sqrt(nb_plots)))
if val_nrows is None: val_nrows = int(np.ceil(nb_plots / val_ncols))

print(nb_plots)
print(val_nrows)
print(val_ncols)

x_val = validation[x_name].to_numpy()
y_val = validation[y_name].to_numpy()

if val_xmin is None: val_xmin = np.floor(x_val.min())
if val_xmax is None: val_xmax = np.ceil(x_val.max())

if val_ymin is None: val_ymin = np.floor(y_val.min())
if val_ymax is None: val_ymax = np.ceil(y_val.max())

if val_xticks is None: val_xticks = get_ticks(val_xmin, val_xmax)
# if val_yticks is None: val_xticks = get_ticks(val_ymin, val_ymax)

# labels for subplots of degrees
validation_titles = [
    f"{fit_labels[degree]} — MAE = {mae(y_val, models[degree](x_val)):.3f} min"
    for degree in degrees
]

validation_figure = make_subplots(
    rows=val_nrows, 
    cols=val_ncols, 
    shared_xaxes="all", 
    shared_yaxes="all",
    subplot_titles=validation_titles, 
    horizontal_spacing=0.11, 
    vertical_spacing=0.16
)

for index, degree in enumerate(degrees):
    add_prediction_view(
        validation_figure, 
        validation,
        x_val, 
        y_val,
        models[degree], 
        x_grid,
        fit_colors[degree],
        # row=index // 2 + 1, 
        # col=index % 2 + 1
        row = index // val_ncols + 1,
        col = index % val_ncols + 1
    )

validation_figure.update_xaxes(
    title_text=dic_label_name[x_name], 
    range=[fit_xmin, fit_xmax],
    tickvals=val_xticks, 
    showticklabels=True
    )

validation_figure.update_yaxes(
    title_text=dic_label_name[y_name], 
    range=[val_ymin, val_ymax],
    showticklabels=True
    )

validation_figure.update_layout(
    title=val_title + f" — {len(x_val)} data points",
    height=850, 
    template="plotly_white", 
    margin={"t": 100, "b": 60}, 
    font={"size": 15}
)

validation_figure.show()

4
2
2


In [246]:
rows = []
for degree, model in models.items():
    rows.append({
        "degree": degree, 
        "model": fit_labels[degree],
        "train_mae": mae(y_train, model(x_train)),
        "validation_mae": mae(validation[y_name], model(validation[x_name])),
    })
comparison = pd.DataFrame(rows)

display(comparison.round(3))

,degree,model,train_mae,validation_mae
0,0,Constante,14.661,13.677
1,1,Droite,3.569,3.264
2,2,Degré 2,1.739,2.056
3,4,Degré 4,1.815,2.173


## Validation plots: Errors

In [247]:
error_plot = comparison.melt(
    id_vars=["degree", "model"], 
    value_vars=["train_mae", "validation_mae"],
    var_name="dataset", 
    value_name=f"mae - {y_name}"
    )

# unpack comparison :

# from comparison =
# 	degree	model	train_mae	validation_mae
# 0	 0	 Constante	14.661292	13.677046
# 1	 1	 Droite	     3.568806	 3.263745
# 2	 2	 Degré 2	 1.739116	 2.055740
# 3	 12	 Degré 12	 0.883025	 3.279982

# to error_plot =
# 	degree	model	     dataset	mae - preparation_minutes
# 0	 0	  Constante	    train_mae	    14.661292
# 1	 1	  Droite	    train_mae	     3.568806
# 2	 2	  Degré 2	    train_mae	     1.739116
# 3	 12	  Degré 12	    train_mae	     0.883025
# 4	 0	  Constante	    validation_mae	13.677046
# 5	 1	  Droite	    validation_mae	 3.263745
# 6	 2	  Degré 2	    validation_mae	 2.055740
# 7	 12	  Degré 12	    validation_mae	 3.279982

error_plot["dataset"] = \
    error_plot["dataset"].map(
        {
            "train_mae": "Train", 
            "validation_mae": "Validation"
        }
    )

# replace names in error_plot["dataset"]

fig = px.bar(
    error_plot, 
    x="model", 
    y=f"mae - {y_name}", 
    color="dataset", 
    barmode="group",
    labels={
        "model": "Model", 
        f"mae - {y_name}": f"MAE ({y_name})", 
        "dataset": "Data"
    },
    title="Evaluate MAE on train and validation")

fig.show()

## Freeze selection using min(MAE on validation_dataset)

In [248]:
# Freeze the selection using validation only, before loading the final test.
selected_row = comparison.loc[comparison["validation_mae"].idxmin()]
chosen_degree = int(selected_row["degree"])
chosen_model = models[chosen_degree]

print("Chosen model :", fit_labels[chosen_degree])

Chosen model : Degré 2


# Final test on disjoint dataset

## Final Test: MAE of chosen model

In [249]:
test_predictions = chosen_model(test[x_name])
final_mae = mae(test[y_name], test_predictions)
print(f"MAE of chosen model : {final_mae:.3f} ({y_name})")

MAE of chosen model : 1.888 (preparation_minutes)


## Final Test: Plot MAE on test data

In [ ]:
# Final test plots

x_test = test[x_name].to_numpy()
y_test = test[y_name].to_numpy()

if test_xmin is None: test_xmin = np.floor(x_test.min())
if test_xmax is None: test_xmax = np.ceil(x_test.max())

if test_ymin is None: test_ymin = np.floor(y_test.min())
if test_ymax is None: test_ymax = np.ceil(y_test.max())

if test_xticks is None: test_xticks = get_ticks(test_xmin, test_xmax)

test_figure = make_subplots(rows=1, cols=1)

add_prediction_view(
    test_figure, 
    test, 
    test[x_name], 
    test[y_name],
    chosen_model, 
    x_grid,
    fit_colors[degree],
    )

test_figure.update_xaxes(
    title_text=dic_label_name[x_name], 
    range=[test_xmin, test_xmax],
    tickvals=test_xticks, 
    showticklabels=True
    )

test_figure.update_yaxes(
    title_text=dic_label_name[y_name], 
    range=[test_ymin, test_ymax],
    showticklabels=True
    )

test_figure.update_layout(
    title="" +
        f"{test_title} — " +
        f"{fit_labels[chosen_degree]} — " +
        f"MAE = {final_mae:.3f} — " +
        f"{len(x_test)} data points",
    height=580, 
    template="plotly_white", 
    font={"size": 16})

test_figure.show()



## Final Test: Plot of Residues

In [251]:
test_residuals = test.assign(
    prediction=test_predictions,
    residual=test[y_name] - test_predictions
)

fig = px.scatter(
    test_residuals, 
    x="prediction", 
    y="residual", 
    hover_data=[id_name, x_name],
    labels={
        "prediction": f"Prediction ({y_name})", 
        "residual": f"Residues ({y_name})"
    },
    title="Dispersion of Residues on Final Test dataset")

fig.add_hline(y=0, line_dash="dash")
fig.show()